# Hansen Ch.28 Model Selection, Stein Shrinkage, and Model Averaging

**Chapter 28**（书稿 PDF 约 **p927–929**，习题 **28.1–28.12**）

理论 step-by-step 见 `Hansen_Ch28_Exercises_Solutions.md`。

本 notebook 复现 **28.12**：Hispanic women 子样本（$n=3003$）上与 §28.18 相同的九个 log 工资模型，比较 **BIC / AIC / CV / FIC\***。


## Exercise 28.12　Hispanic women：经验回报模型选择

设定与 Table 28.1 平行：
- 样本：`cps09mar`，`female==1` 且 `hisp==1`，$n=3003$
- $Y=\log(\mathrm{earnings}/(\mathrm{hours}\times\mathrm{week}))$
- 共同控制：已婚（`marital==1`）、地区虚拟（相对 region=1）
- 教育：college 虚拟 / 结点在 9 的线性样条 / 学历虚拟
- 经验：$\mathrm{exp}=\mathrm{age}-\mathrm{educ}-6$ 的 2/4/6 次多项式
- **焦点参数**：0 与 30 年经验的 log 工资差 $\Delta$（并报告 $100(e^\Delta-1)$ 百分数）


In [ ]:
# Hansen Ch.28 — 模型选择实证（详尽注释）
import numpy as np
import pandas as pd
from pathlib import Path

ROOT = Path("../..") / "hansen" / "econometrics" / "data"  # relative to docs/chXX/
cps = pd.read_stata(ROOT / "cps09mar" / "cps09mar.dta")

# ---- 样本：Hispanic women ----
df = cps.loc[(cps["female"] == 1) & (cps["hisp"] == 1)].copy()
wage = df["earnings"].to_numpy(float) / (df["hours"].to_numpy(float) * df["week"].to_numpy(float))
ok = np.isfinite(wage) & (wage > 0)
df = df.loc[ok].copy()
y = np.log(wage[ok])
educ = df["education"].to_numpy(float)
exp = df["age"].to_numpy(float) - educ - 6.0  # 潜在经验
married = (df["marital"].to_numpy() == 1).astype(float)
reg = df["region"].to_numpy()
R2 = (reg == 2).astype(float)
R3 = (reg == 3).astype(float)
R4 = (reg == 4).astype(float)
n = len(y)
print(f"n = {n}  (教材：3003)")

def design(ed_type: str, exp_pow: int) -> np.ndarray:
    """构造设计矩阵：截距+已婚+地区 + 教育块 + 经验多项式。"""
    base = [np.ones(n), married, R2, R3, R4]
    if ed_type == "college":
        E = [(educ >= 16).astype(float)]
    elif ed_type == "spline":
        # 线性样条，结点 educ=9：educ 与 (educ-9)+
        E = [educ, np.maximum(educ - 9.0, 0.0)]
    else:  # dummy for 12,13,14,16,18,20
        E = [(educ == e).astype(float) for e in (12, 13, 14, 16, 18, 20)]
    P = [exp ** k for k in range(1, exp_pow + 1)]
    return np.column_stack(base + E + P)


def ols_metrics(X, y):
    """OLS + BIC/AIC/CV + HC1 协方差。"""
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ b
    n, k = X.shape
    sse = float(e @ e)
    sigma2 = sse / n
    # leave-one-out CV：tilde e_i = e_i/(1-h_ii)
    XtX_inv = np.linalg.inv(X.T @ X)
    h = np.sum((X @ XtX_inv) * X, axis=1)
    cv = float(np.sum((e / (1.0 - h)) ** 2))
    bic = n * np.log(sigma2) + k * np.log(n)
    aic = n * np.log(sigma2) + 2 * k
    # HC1
    meat = X.T @ (X * (e ** 2)[:, None])
    V = XtX_inv @ meat @ XtX_inv * (n / (n - k))
    return b, V, k, bic, aic, cv


def focus_delta(b, ed_type, exp_pow):
    """焦点：经验 0→30 的 log 工资差（教育/其他控制相同则抵消）。"""
    n_base, n_ed = 5, {"college": 1, "spline": 2, "dummy": 6}[ed_type]
    idx = list(range(n_base + n_ed, n_base + n_ed + exp_pow))
    d = sum(b[i] * (30.0 ** (j + 1)) for j, i in enumerate(idx))
    g = np.zeros(len(b))
    for j, i in enumerate(idx):
        g[i] = 30.0 ** (j + 1)
    return d, g


specs = [
    (1, "college", 2), (2, "spline", 2), (3, "dummy", 2),
    (4, "college", 4), (5, "spline", 4), (6, "dummy", 4),
    (7, "college", 6), (8, "spline", 6), (9, "dummy", 6),
]

# 无约束（最大）模型 = Model 9，用于 FIC 的 mu_hat
X9 = design("dummy", 6)
b9, V9, k9, *_ = ols_metrics(X9, y)
mu_hat, _ = focus_delta(b9, "dummy", 6)

rows = []
print(f"{'M':>2} {'educ':>8} {'p':>2} {'100Δ':>7} {'se':>6} {'ret%':>7} {'BIC':>9} {'AIC':>9} {'CV':>8} {'FIC*':>8}")
for mid, ed, p in specs:
    X = design(ed, p)
    b, V, k, bic, aic, cv = ols_metrics(X, y)
    d, g = focus_delta(b, ed, p)
    se = float(np.sqrt(g @ V @ g))
    ret = 100 * (np.exp(d) - 1)  # 期望工资的百分差
    fic = n * (d - mu_hat) ** 2 + 2 * n * (se ** 2)  # (28.20) 型 FIC*
    print(f"{mid:2d} {ed:>8} {p:2d} {100*d:7.1f} {100*se:6.1f} {ret:7.1f} {bic:9.1f} {aic:9.1f} {cv:8.1f} {fic:8.1f}")
    rows.append(dict(mid=mid, ed=ed, p=p, d=100*d, se=100*se, ret=ret, bic=bic, aic=aic, cv=cv, fic=fic))

print("\n选择结果：")
for name, key in [("BIC", "bic"), ("AIC", "aic"), ("CV", "cv"), ("FIC*", "fic")]:
    best = min(rows, key=lambda r: r[key])
    print(f"  {name:4s} → Model {best['mid']} ({best['ed']}, exp^{best['p']}): "
          f"100Δ≈{best['d']:.1f}, ret%≈{best['ret']:.1f}")

print("""
解读（对照亚洲女性 §28.18）：
- BIC 偏好更省的 Model 2（样条教育 + 二次经验）。
- AIC / CV / FIC* 一致选 Model 5（样条 + 4 次经验）——经验非线性更丰富。
- 估计对教育设定很敏感（college-only 明显偏低）；经验阶数从 2→4 抬高回报，6 次收益有限。
- 首选建议：以 CV/AIC/FIC 的 Model 5 为主报告，并展示 Model 2 vs 5/9 的稳健性。
""")
